The goal of this assignment is to visit news.asu.edu, scrape current URLs (in a /YYYYMMDD pattern), search these stories for keywords, and append the found keywords and corresponding URLs to found_hits.csv.  

#Step 1: Load needed modules:

In [2]:
import os
import re, csv, requests
from bs4 import BeautifulSoup #parses HTML
from urllib.parse import urljoin

Here we are going to do a set up, then "call" or fetch the ASU News home page, and make sure it answers (returns bytes) before continuing.

#Step 2:  Establish connection w/ ASU news:

In [4]:
BASE = "https://news.asu.edu/"
HEADERS = {"User-Agent": "ASU-News-Monitor/1.0 (contact: salandon@asu.edu)"}
#DELAY = 1.5 #polite pause, in script
#TIMEOUT = 15 #give up if website doesn't reply, in script
CSV_PATH = "found_hits.csv" #where the hits are being saved

KEYWORDS = ["medicine", "breakthrough", "upcoming", "innovation", "predictive", "health"]

#confirm working:

r = requests.get(BASE, headers=HEADERS, timeout=15) #function from requests to send get
r.raise_for_status() #r = response object, stop if error
print("homepage bytes:", len(r.text)) #confirm download


homepage bytes: 145594


The "connection" appears to be working, so we will continue.  This section is going to find the current article URLs (which follow 8 digit date) on the home page, then pull the hrefs, using regex to get valid news articles.  It parses using BeautifulSoup, then collects a tags with the href attribute and extracts.  Since it cleans the URLs, filters for only articles, removes duplicates, it ends with a clean list of URLs for scraping.

#Step 3:  Pull URLs of articles for scraping:

In [8]:
soup = BeautifulSoup(r.text, "html.parser") #parses the homepage HTML
links = set() #create empty set for the links (set removes dupes)

for a in soup.select("a[href]"):  #function to pull all a tags w/ href attribute
    href = a.get("href")     #get (extract) href
    if not href:
        continue
    href = urljoin(BASE, href) #convert URLs to full URLs (debugs)

    # keep only actual article URLs with /YYYYMMDD
    if href.startswith(BASE) and re.search(r"/\d{8}-", href):  #makes sure it is an article, regex search article prefix
        links.add(href.split("#")[0]) #removes id fragments

links = sorted(links) #sorts aphabetically
print("found links:", len(links)) #tell us how many links it found
links[:5] #print the first 5 URLs to examine before continuing

found links: 35


['https://news.asu.edu/20250807-environment-and-sustainability-preparing-drier-future-colorado-river-basin',
 'https://news.asu.edu/20250915-science-and-technology-asu-student-works-protect-african-penguins-local-aquarium',
 'https://news.asu.edu/20251003-business-and-entrepreneurship-new-study-shows-how-hospitals-buy-supplies-could-make-or',
 'https://news.asu.edu/20251009-local-national-and-global-affairs-adaptability-key-future-public-administration',
 'https://news.asu.edu/20251017-sun-devil-community-100m-renovation-energize-fan-experience-desert-financial-arena']

That looks good so far.  Next, for each article I want to pull the story body so we can search for keywords.  I'll put in a polite sleep, download the HTML, parse, try to find the main article and return.

#Step 4:  Pull the story text to search for keywords:

In [10]:
def fetch_article_text(url):   #Fetch an article and return its text content as a single string.

    # Polite pause
    time.sleep(1.5)

    # Download article HTML
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()  # stop immediately if the article didn't load properly, debug

    soup = BeautifulSoup(r.text, "html.parser")

    # Try to find the main article
    body_el = (
        soup.select_one("article") or
        soup.select_one(".field--name-body, .story, .content, main")
    )

    # Return extracted text content
    if body_el:
        return body_el.get_text(" ", strip=True)
    else:
        # Fallback: return all text if article structure is unusual
        return soup.get_text(" ", strip=True)

#Test on the first link to verify
test_article = fetch_article_text(links[0])
print("Article length:", len(test_article))
print(test_article[:500], "...")


Article length: 11882
Environment and sustainability Preparing for a drier future on the Colorado River basin With a looming deadline for the Colorado River Compact, ASU water experts weigh in on the state's water forecast Lake Pleasant (pictured), located north of Phoenix, serves as the Central Arizona Project’s water storage reservoir, as well as being a popular recreational amenity. Water shortages are impacting Colorado River basin reservoirs such as Lake Mead in Nevada and Lake Powell, which stretches across nor ...


Next I need to loop over URLs that have been extracted, fetch the article text, search for keywords, and record a hit when we get one.

# Step 5: Search each article for keyword hits and record:

In [11]:
# compile whole word, case insensitive patterns
patterns = [(kw, re.compile(rf"\b{re.escape(kw)}\b", re.IGNORECASE)) for kw in KEYWORDS]

hits = []  # collect keyword, url pairs here

# loop over article URLs and search for keywords
for url in links:
    text = fetch_article_text(url)   # polite delay is in here
    if not text:
        continue

    for kw, rx in patterns:
        if rx.search(text):
            hits.append([kw, url])

# sanity check
print("Total hits found:", len(hits))
hits[:3]  # show first 3 hits


Total hits found: 56


[['innovation',
  'https://news.asu.edu/20250807-environment-and-sustainability-preparing-drier-future-colorado-river-basin'],
 ['health',
  'https://news.asu.edu/20250915-science-and-technology-asu-student-works-protect-african-penguins-local-aquarium'],
 ['innovation',
  'https://news.asu.edu/20251003-business-and-entrepreneurship-new-study-shows-how-hospitals-buy-supplies-could-make-or']]

This chunk is the writing to found_hits.csv chunk.

#Step 6:  Append hits to found_hits.csv with a header

In [12]:
#CSV_PATH = "found_hits.csv" previously saved

# Create the CSV with header
if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["keyword", "url"])

# Prepare keyword, url pairs for this run

wrote = 0
with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    for kw, url in hits:
        writer.writerow([kw, url])
        wrote += 1

print(f"Appended {wrote} rows to {CSV_PATH}")

Appended 56 rows to found_hits.csv
